In [2]:
import tkinter as tk
from tkinter import filedialog, messagebox
import cv2
import os
from PIL import Image, ImageTk

In [12]:
class LabelImgApp:
    def __init__(self, master):
        self.master = master
        master.title("LabelImg for YOLO")

        self.canvas = tk.Canvas(master, width=800, height=600)
        self.canvas.pack()

        self.img = None
        self.filepath = None
        self.bboxes = []
        self.current_bbox = None
        self.selected_bbox = None
        self.classes = []
        self.current_class_index = 0

        self.btn_load = tk.Button(master, text="Load Image", command=self.load_image)
        self.btn_load.pack()
        self.btn_save = tk.Button(master, text="Save Labels", command=self.save_labels)
        self.btn_save.pack()

        self.class_label = tk.Label(master, text="Current Class: ")
        self.class_label.pack()
        self.class_entry = tk.Entry(master)
        self.class_entry.pack()
        self.class_entry.insert(0, "Class0")
        self.btn_next_class = tk.Button(master, text="Next Class", command=self.next_class)
        self.btn_next_class.pack()

        self.canvas.bind("<Button-1>", self.handle_bbox)
        self.canvas.bind("<B1-Motion>", self.draw_bbox)
        self.canvas.bind("<ButtonRelease-1>", self.end_bbox)
        self.canvas.bind("<Delete>", self.delete_bbox)

    def load_image(self):
        self.filepath = filedialog.askopenfilename()
        if self.filepath:
            self.img = cv2.imread(self.filepath)
            # Reset các bounding box và class hiện tại khi load ảnh mới
            self.bboxes = []  
            self.current_class_index = 0
            self.class_label.config(text=f"Current Class: Class0")
            self.show_image()

    def show_image(self):
       if self.img is not None:
            # Lấy kích thước của canvas
            canvas_width = self.canvas.winfo_width()
            canvas_height = self.canvas.winfo_height()

            # Tính toán tỷ lệ để resize ảnh
            img_height, img_width, _ = self.img.shape
            ratio = min(canvas_width / img_width, canvas_height / img_height)

            # Resize ảnh
            new_width = int(img_width * ratio)
            new_height = int(img_height * ratio)
            resized_img = cv2.resize(self.img, (new_width, new_height))

            # Chuyển đổi sang định dạng tkinter
            img_rgb = cv2.cvtColor(resized_img, cv2.COLOR_BGR2RGB)
            img_pil = Image.fromarray(img_rgb)
            img_tk = ImageTk.PhotoImage(img_pil)

            # Hiển thị ảnh trên canvas
            self.canvas.create_image(0, 0, anchor=tk.NW, image=img_tk)
            self.canvas.image = img_tk  # Keep a reference to avoid garbage collection
            self.master.after(100, self.create_photo_image, img_pil)
    def create_photo_image(self, img_pil):
            img_tk = ImageTk.PhotoImage(img_pil)
            self.canvas.create_image(0, 0, anchor=tk.NW, image=img_tk)
            self.canvas.image = img_tk
    def convert_to_tk(self, img):
        import PIL.Image, PIL.ImageTk
        img_pil = PIL.Image.fromarray(img)
        return PIL.ImageTk.PhotoImage(img_pil)

    def handle_bbox(self, event):
        for i, bbox in enumerate(self.bboxes):
            x1, y1, x2, y2 = bbox["coords"]
            if x1 <= event.x <= x2 and y1 <= event.y <= y2:
                self.selected_bbox = i
                self.canvas.itemconfig(self.current_bbox, outline="green")
                return  # Dừng xử lý nếu đã chọn được bounding box

        # Nếu không chọn được bounding box nào, bắt đầu vẽ bounding box mới
        self.start_bbox(event)

    def start_bbox(self, event):
        self.current_bbox = self.canvas.create_rectangle(event.x, event.y, event.x, event.y, outline="red")
        self.bboxes.append({"coords": [event.x, event.y, event.x, event.y], "class": self.current_class_index})
        self.selected_bbox = None  # Bỏ chọn bounding box nếu đang chọn

    def draw_bbox(self, event):
        if self.current_bbox:
            x1, y1, x2, y2 = self.canvas.coords(self.current_bbox)
            self.canvas.coords(self.current_bbox, x1, y1, event.x, event.y)
            self.bboxes[-1]["coords"] = [x1, y1, event.x, event.y]  # Cập nhật tọa độ trong danh sách bboxes

    def end_bbox(self, event):
        if self.current_bbox:  # Chỉ thực hiện khi có bounding box đang vẽ
            x1, y1, x2, y2 = self.canvas.coords(self.current_bbox)
            self.bboxes[-1]["coords"] = [x1, y1, x2, y2]  # Cập nhật tọa độ cuối cùng
        self.current_bbox = None  # Reset current_bbox sau khi vẽ xong

    def delete_bbox(self, event):
        if self.selected_bbox is not None:
            self.canvas.delete(self.current_bbox)
            del self.bboxes[self.selected_bbox]
            self.selected_bbox = None

    def save_labels(self):
        if self.img is not None:
            # Hỏi người dùng chọn thư mục để lưu file nhãn
            label_dir = os.getcwd() + "/dataset/train/Labels"
            if label_dir:
                filename = os.path.splitext(os.path.basename(self.filepath))[0] + ".txt"
                label_path = os.path.join(label_dir, filename)
                with open(label_path, "w") as f:
                    for bbox in self.bboxes:
                        x_center = (bbox["coords"][0] + bbox["coords"][2]) / 2 / self.img.shape[1]
                        y_center = (bbox["coords"][1] + bbox["coords"][3]) / 2 / self.img.shape[0]
                        width = (bbox["coords"][2] - bbox["coords"][0]) / self.img.shape[1]
                        height = (bbox["coords"][3] - bbox["coords"][1]) / self.img.shape[0]
                        class_index = bbox["class"]
                        f.write(f"{class_index} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")
                # Xóa ảnh và bounding box sau khi lưu
                self.canvas.delete("all")
                self.img = None
                self.filepath = None
                self.bboxes = []
                self.bbox_rects = []
                messagebox.showinfo("Success", f"Labels saved to {label_dir}")
        else:
            messagebox.showerror("Error", "No image loaded")

    def next_class(self):
        new_class = self.class_entry.get()
        if new_class not in self.classes:
            self.classes.append(new_class)
        self.current_class_index = self.classes.index(new_class)
        self.class_label.config(text=f"Current Class: {new_class}")

In [13]:
root = tk.Tk()
app = LabelImgApp(root)
root.mainloop()